# 02 — Build the offering table

One row per Reg CF offering (`FILE_NUMBER`) with its terms, both targets and data-quality flags, saved to `data/processed/offerings.parquet` so later notebooks load it instead of re-reading the raw zips.

Nothing is dropped here: implausible terms are only flagged (`TARGET_SUSPECT`, `DEADLINE_SUSPECT`), and each analysis notebook decides its sample.

**Input:** `data/raw/` filled by `python -m src.download`. **Output:** `data/processed/offerings.parquet`. **Runtime:** ~1–2 min, < 4 GB RAM.

In [1]:
import sys
from pathlib import Path

# Repo root: the folder holding src/, searched upwards from the working
# directory, then one level down (kernels do not always start in notebooks/).
here = Path.cwd().resolve()
candidates = [here, *here.parents, *(p for p in here.iterdir() if p.is_dir())]
ROOT = next((p for p in candidates if (p / "src" / "cf_data.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError(f"Repo root (folder with src/cf_data.py) not found from {here}")
sys.path.insert(0, str(ROOT))

import pandas as pd

from src import cf_data as cf
from src.formd_data import filings_by_cik
from src.targets import MAX_CAMPAIGN_DAYS, MIN_TARGET, add_annual_reports, add_formd, build_offerings

In [2]:
sub = cf.read_table("SUBMISSION")
dis = cf.read_table("DISCLOSURE")
iss = cf.read_table("ISSUER_INFORMATION")
off = build_offerings(sub, dis, iss)
off = add_annual_reports(off, sub)
off = add_formd(off, filings_by_cik())
print(f"{len(off):,} offerings, {off.shape[1]} columns")

11,122 offerings, 17 columns


## Data-quality flags

In [3]:
print(f"TARGET_SUSPECT (target < ${MIN_TARGET:,} or missing): {off.TARGET_SUSPECT.sum()}")
print(f"DEADLINE_SUSPECT (missing, before the Form C, or > {MAX_CAMPAIGN_DAYS} days after): {off.DEADLINE_SUSPECT.sum()}")
off.loc[off.TARGET_SUSPECT | off.DEADLINE_SUSPECT, ["CIK", "C_DATE", "DEADLINE", "TARGET", "Y_SOURCE"]]

TARGET_SUSPECT (target < $1,000 or missing): 26
DEADLINE_SUSPECT (missing, before the Form C, or > 1095 days after): 32


,CIK,C_DATE,DEADLINE,TARGET,Y_SOURCE
FILE_NUMBER,,,,,
020-23334,0001674612,2017-04-28,2021-07-31,10000.00,C-U success
020-23669,0001139685,2017-10-13,2025-08-09,10000.00,C-U success
020-24159,0001694455,2018-04-11,2018-04-10,100000.00,<NA>
020-24160,0001736388,2018-04-11,2024-03-22,10000.00,C-U success
020-24293,0001736664,2018-05-23,2022-05-31,10000.00,C-U success
020-24319,0001702607,2018-05-30,2022-12-31,10000.00,<NA>
020-24486,0001743601,2018-07-13,2022-01-05,10000.00,<NA>
020-24843,0001740075,2018-10-29,2022-02-25,10000.00,<NA>
020-25316,0001773139,2019-04-16,2022-11-01,10000.00,C-U success


## Targets

In [4]:
print("Y_SUCCESS sources:", off.Y_SOURCE.value_counts(dropna=False).to_dict())
w = off[off.FD_WINDOW_COMPLETE]
print(f"Y_FORMD: {w.Y_FORMD.mean():.1%} of {len(w):,} offerings with a complete 24-month window")

Y_SUCCESS sources: {<NA>: 5198, 'C-U success': 4559, 'withdrawn': 1041, 'C-U final failure': 324}
Y_FORMD: 8.3% of 8,417 offerings with a complete 24-month window


## Save

In [5]:
out = ROOT / "data" / "processed" / "offerings.parquet"
off.to_parquet(out)
print(out.relative_to(ROOT), f"{out.stat().st_size / 1e6:.1f} MB")
off.dtypes

data/processed/offerings.parquet 0.4 MB


CIK                           object
ACCESSION_NUMBER              object
C_DATE                datetime64[ns]
TARGET                       float64
DEADLINE              datetime64[ns]
WITHDRAWN                       bool
TARGET_SUSPECT                  bool
DEADLINE_SUSPECT                bool
N_CU                           int64
AMOUNT_RAISED                float64
Y_SUCCESS                    float64
Y_SOURCE              string[python]
AR_AFTER                        bool
FD_BEFORE                       bool
FD_DURING                       bool
Y_FORMD                         bool
FD_WINDOW_COMPLETE              bool
dtype: object